# Medical CPT Training - 8-Bit Quantized
## Qwen2.5-7B with Memory Optimization for 32GB GPUs

**8-bit quantization reduces model size from 15GB → 4GB**

This notebook uses 8-bit quantization which allows training on GPUs with limited VRAM (24-32GB).

**Hardware:**
- RTX 4090, RTX 5090, A100, or similar (24GB+)
- Ubuntu 20.04+ with CUDA 12.1+
- 100GB+ disk space

**Expected Training Time: 12-18 hours** (8-bit is slower than full precision)

## 1. Environment & System Check

In [ ]:
import os
import sys
import json
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List

print("\n" + "="*80)
print("SYSTEM CHECK - 8-BIT QUANTIZED TRAINING")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (FATAL)'}")

if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")
    
    if gpu_mem < 24:
        print(f"\n⚠️  WARNING: GPU only has {gpu_mem:.1f}GB (need ≥24GB for safe training)")
        print("Consider using 4-bit quantization instead.\n")
else:
    print("✗ CUDA not available - training will be impossible\n")
    sys.exit(1)

print("="*80 + "\n")

In [ ]:
# Check required packages
print("📦 Checking packages...")
required = ['transformers', 'datasets', 'torch', 'bitsandbytes', 'peft']

try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        DataCollatorForLanguageModeling,
        BitsAndBytesConfig,
    )
    from datasets import Dataset
    from peft import prepare_model_for_kbit_training
    print("✓ All packages available\n")
except ImportError as e:
    print(f"✗ Missing: {e}")
    print("\nInstall with:")
    print("pip install -q transformers datasets torch bitsandbytes peft")
    sys.exit(1)

## 2. Configuration

In [ ]:
# ============ TRAINING CONFIG (8-BIT OPTIMIZED) ============
CONFIG = {
    # Model & Data
    "model_name": "Qwen/Qwen2.5-7B",
    "train_file": "augmented_output/train.jsonl",
    "eval_file": "augmented_output/eval.jsonl",

    # Training (tuned for 8-bit on 32GB GPU)
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,   # Small per-device batch
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 16,  # Effective batch = 1*16 = 16
    "learning_rate": 2e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,

    # Sequence length reduced to save memory
    "max_seq_length": 1024,  # Was 2048
    "bf16": True,
    "fp16": False,

    # Saving
    "output_dir": "medical_qwen_cpt_8bit",
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,

    # Evaluation
    "eval_strategy": "steps",
    "eval_steps": 50,

    # Logging
    "logging_dir": "logs_8bit",
    "logging_steps": 5,

    # Data loading
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": False,

    "seed": 42,
}

print("\n" + "="*80)
print("CONFIGURATION - 8-BIT QUANTIZED TRAINING")
print("="*80)
print(f"\n⚠️  Memory Optimizations Enabled:")
print(f"  ✓ 8-bit quantization (15GB → 4GB)")
print(f"  ✓ Gradient checkpointing")
print(f"  ✓ Reduced sequence length (1024)")
print(f"  ✓ Small batch size with accumulation")
print(f"\nExpected GPU Memory: ~22-26 GB")
print("\nConfiguration:")
for key, value in sorted(CONFIG.items()):
    print(f"  {key:.<50} {value}")

effective_batch = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]
print(f"\n  Effective batch size: {effective_batch}")
print(f"\n" + "="*80 + "\n")

## 3. Load Data

In [ ]:
def load_jsonl(file_path: str, max_samples: int = None) -> List[Dict]:
    """Load JSONL file."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return data

print("📂 Loading data...")

if not Path(CONFIG["train_file"]).exists():
    print(f"✗ {CONFIG['train_file']} not found!")
    raise FileNotFoundError(f"Run augmentation first: python augment_medical_data.py")

train_data = load_jsonl(CONFIG["train_file"])
eval_data = load_jsonl(CONFIG["eval_file"])

train_tokens = sum(c.get('token_count', 0) for c in train_data)
eval_tokens = sum(c.get('token_count', 0) for c in eval_data)

print(f"✓ Train: {len(train_data):,} chunks ({train_tokens:,} tokens)")
print(f"✓ Eval:  {len(eval_data):,} chunks ({eval_tokens:,} tokens)\n")

In [ ]:
# Show sample
print("Sample training chunk:")
sample = train_data[0]
print(f"  Source: {sample.get('metadata', {}).get('source_book', 'N/A')}")
print(f"  Type: {sample.get('metadata', {}).get('augmentation', 'original')}")
print(f"  Tokens: {sample.get('token_count')}")
print(f"  Text: {sample['text'][:300]}...\n")

## 4. Load Tokenizer

In [ ]:
print(f"🔄 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Tokenizer loaded")
print(f"  Vocab size: {len(tokenizer):,}")
print(f"  Pad token: {tokenizer.pad_token_id}\n")

## 5. Load Model with 8-Bit Quantization

In [ ]:
print(f"🔄 Loading model with 8-bit quantization...")
print("   This saves 70% GPU memory (15GB → 4GB)\n")

# Configure 8-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

# Load with quantization
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# Prepare for training with 8-bit
model = prepare_model_for_kbit_training(model)

print(f"✓ Model loaded")
num_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {num_params/1e9:.2f}B")
print(f"  Quantization: 8-bit")
print(f"  GPU footprint: ~4GB (down from 15GB)")
print(f"  Gradient checkpointing: Enabled\n")

## 6. Tokenize Datasets

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=CONFIG["max_seq_length"],
        padding="max_length",
    )

print(f"🔄 Tokenizing {len(train_data):,} training samples...")
train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

print(f"🔄 Tokenizing {len(eval_data):,} eval samples...")
eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})
eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

print(f"\n✓ Tokenization complete")
print(f"  Train: {len(train_dataset):,} samples")
print(f"  Eval:  {len(eval_dataset):,} samples\n")

## 7. Setup Training

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    max_grad_norm=CONFIG["max_grad_norm"],
    bf16=CONFIG["bf16"],
    fp16=CONFIG["fp16"],
    save_strategy=CONFIG["save_strategy"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    eval_strategy=CONFIG["eval_strategy"],
    eval_steps=CONFIG["eval_steps"],
    metric_for_best_model="eval_loss",
    logging_dir=CONFIG["logging_dir"],
    logging_steps=CONFIG["logging_steps"],
    seed=CONFIG["seed"],
    dataloader_num_workers=CONFIG["dataloader_num_workers"],
    dataloader_pin_memory=CONFIG["dataloader_pin_memory"],
    load_best_model_at_end=True,
    greater_is_better=False,
    push_to_hub=False,
    report_to=["tensorboard"],
)

print("✓ Training arguments configured\n")

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✓ Trainer created")
print(f"\n📊 Training Summary:")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Quantization: 8-bit (4GB footprint)")
print(f"  Training samples: {len(train_dataset):,}")
print(f"  Eval samples: {len(eval_dataset):,}")
print(f"  Epochs: {CONFIG['num_train_epochs']}")
print(f"  Effective batch: {effective_batch}")
print(f"  Expected time: 12-18 hours")
print(f"\n💾 Output: {CONFIG['output_dir']}/")
print(f"📊 Logs: {CONFIG['logging_dir']}/")
print(f"\n" + "="*80)
print("⚠️  TRAINING STARTING - Press Ctrl+C to cancel")
print("="*80 + "\n")

## 8. Train Model

In [ ]:
print(f"🚀 Starting training: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    train_result = trainer.train()
    print(f"\n✓ Training complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Training loss: {train_result.training_loss:.4f}\n")
except KeyboardInterrupt:
    print("\n⏹️  Training interrupted by user")
except Exception as e:
    print(f"\n✗ Training failed: {e}")
    import traceback
    traceback.print_exc()

## 9. Evaluate

In [ ]:
print("🔍 Evaluating model...\n")
eval_results = trainer.evaluate()

print("📊 Results:")
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
print()

## 10. Save Model

In [ ]:
best_model_path = Path(CONFIG["output_dir"]) / "best_model"
print(f"💾 Saving model to {best_model_path}...")

best_model_path.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

model_size = sum(f.stat().st_size for f in best_model_path.glob('**/*')) / 1e9
print(f"✓ Saved ({model_size:.2f} GB)\n")

## 11. Test Inference

In [ ]:
print("🧪 Testing inference...\n")

from transformers import pipeline

inference_model = AutoModelForCausalLM.from_pretrained(
    str(best_model_path),
    quantization_config=bnb_config,
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=inference_model,
    tokenizer=tokenizer,
    device=0,
)

prompts = [
    "The pathophysiology of venous insufficiency involves",
    "Duplex ultrasound is important for",
]

for prompt in prompts:
    output = generator(
        prompt,
        max_length=100,
        temperature=0.7,
        do_sample=True,
    )
    print(f"Prompt: {prompt}")
    print(f"Output: {output[0]['generated_text']}\n")

## 12. Summary

In [ ]:
print("\n" + "="*80)
print("✓ TRAINING COMPLETE")
print("="*80)
print(f"\nBest model: {best_model_path}")
print(f"Size: {model_size:.2f} GB")
print(f"\nEvaluation:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
print(f"\n" + "="*80)